In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

RAW_DATA = Path("../data/raw")  # use Path("data/raw") if running from project root

mips = pd.read_csv(RAW_DATA / "grp_public_reporting.csv", dtype=str)
dac = pd.read_csv(RAW_DATA / "DAC_NationalDownloadableFile.csv", dtype=str)

# Clean column names because MIPS has leading spaces
mips.columns = mips.columns.str.strip()
dac.columns = dac.columns.str.strip()

# Make join keys consistent
mips["org_pac_id"] = mips["org_PAC_ID"].astype(str).str.strip()
dac["org_pac_id"] = dac["org_pac_id"].astype(str).str.strip()

In [3]:
def clean_zip5(x):
    if pd.isna(x):
        return pd.NA
    
    x = str(x).strip()
    
    # Remove decimal artifact like "602.0"
    if x.endswith(".0"):
        x = x[:-2]
    
    # Keep only digits
    digits = "".join(ch for ch in x if ch.isdigit())
    
    if len(digits) == 0:
        return pd.NA
    
    # If ZIP+4 or longer, keep first 5
    # If short like Puerto Rico 602, pad to 00602
    return digits[:5].zfill(5)

dac["zip5"] = dac["ZIP Code"].apply(clean_zip5)

dac[["org_pac_id", "ZIP Code", "zip5"]].head()

,org_pac_id,ZIP Code,zip5
0,nan,602,00602
1,nan,602,00602
2,nan,622,00622
3,nan,646,00646
4,6305731118,646,00646


In [ ]:
# Clean org_pac_id column in dac, remove na values (individual physicians have no org_pac_id)

dac["org_pac_id"] = dac["org_pac_id"].astype("string").str.strip()

dac_group = dac[
    dac["org_pac_id"].notna() &
    dac["org_pac_id"].ne("") &
    dac["org_pac_id"].str.lower().ne("nan")
].copy()

dac_group[["org_pac_id", "ZIP Code", "zip5"]].head()

,org_pac_id,ZIP Code,zip5
4,6305731118,646,00646
17,9638151756,802,00802
18,2860304482,840,00840
19,4385740141,840,00840
20,2264424712,907,00907
